In [3]:
import pandas as pd
import pull_from_huggingface
import random
import utils
import openai_interface as openai
import generate_data as gd

In [17]:
PROVIDER = 'openai'
MODEL = 'gpt-4o-mini-2024-07-18'

In [4]:
NLP_INPUTS_TO_LABELS = pull_from_huggingface.collect_all_datasets();
print(f'#Downloaded samples: {len(NLP_INPUTS_TO_LABELS)}')

#Downloaded samples: 24227


Choose a subset of the loaded data

In [12]:
N = 10

NLP_INPUTS_TO_LABELS_SUBSET = dict(random.sample(list(NLP_INPUTS_TO_LABELS.items()), N))

df_subset = pd.DataFrame.from_dict(NLP_INPUTS_TO_LABELS_SUBSET, orient='index', columns=['label'])
df_subset.reset_index(inplace=True)
df_subset.columns = ['input', 'label']
print(df_subset.head())

df_subset.to_csv('data_source_nlp/input_label_pairs_unfiltered.csv', index=False)

                                               input                  label
0  "My dogs like to be nice and toasty while rela...              Not Irony
1    "What did you say (that) the poet had written?"    Acceptable Sentence
2  "The more he reads, the more people I wonder w...  Unacceptable Sentence
3  "the result puts a human face on derrida , and...     Positive Sentiment
4  "Haha love when I accidentally spray perfume i...                  Irony


# Filter out pairs for which the model does not know the answer

In [18]:
if PROVIDER == 'openai':
    get_completion = openai.get_completion
else:
    raise ValueError(f'Unknown provider: {PROVIDER}')

In [30]:
def filter_data(df_unfiltered: pd.DataFrame) -> pd.DataFrame:
    """
    Filter out pairs for which the model does not know the answer
    """
    if 'input' not in df_unfiltered.columns or 'label' not in df_unfiltered.columns:
        raise ValueError("DataFrame must contain 'input' and 'label' columns")

    dict_unfiltered = gd.generate_data_for_filtering(dict(zip(df_unfiltered['input'], df_unfiltered['label'])))
    can_be_used = []

    responses = []
    for prompt, expected_answer in dict_unfiltered.items():
        response = get_completion(prompt, model=MODEL)
        responses.append(response)

        if response != expected_answer:
            can_be_used.append(False)
        else:
            can_be_used.append(True)

    print(f'{can_be_used.count(True) / len(can_be_used) * 100:.2f}% of the data can be used')

    df_filtered = df_unfiltered.copy()
    # Add these three columns as a sanity check, so that we can compare the expected and actual answers
    df_filtered['prompt'] = list(dict_unfiltered.keys())
    df_filtered['expected_answer'] = list(dict_unfiltered.values())
    df_filtered['actual_answer'] = responses
    # Filter out the data
    df_filtered = df_filtered[can_be_used]
    df_filtered.to_csv('data_source_nlp/input_label_pairs_filtered.csv', index=False)

    #train/test split
    df_train = df_filtered.sample(frac=0.8, random_state=42)
    df_test = df_filtered.drop(df_train.index)

    df_train.to_csv('data_source_nlp/input_label_pairs_filtered_train.csv', index=False)
    df_test.to_csv('data_source_nlp/input_label_pairs_filtered_test.csv', index=False)

    return df_filtered


This gives us the filtered out dataset:

In [31]:
df_filtered = filter_data(df_subset)

50.00% of the data can be used


,input,label,prompt,expected_answer,actual_answer
0,"""My dogs like to be nice and toasty while rela...",Not Irony,Do you agree or disagree with the following s...,(A),(A)
1,"""What did you say (that) the poet had written?""",Acceptable Sentence,Do you agree or disagree with the following s...,(B),(B)
2,"""The more he reads, the more people I wonder w...",Unacceptable Sentence,Do you agree or disagree with the following s...,(B),(B)
5,"""while puerile men dominate the story , the wo...",Positive Sentiment,Do you agree or disagree with the following s...,(A),(A)
9,"""We 're confident that the new leadership will...",Equivalent,Do you agree or disagree with the following s...,(A),(A)
